# OCR Accuracy Evaluation

Compare OCR results against ground truth data to evaluate accuracy.

In [17]:
from __future__ import annotations

from pathlib import Path
import sys

# Find project root by looking for src/ directory
project_root = Path.cwd()
while not (project_root / 'src').exists() and project_root.parent != project_root:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import json
from typing import Any, Dict, List, Optional, Tuple

BASE_DIR = project_root
OCR_DIR = BASE_DIR / "outputs/processed/json_data"
TRUTH_DIR = BASE_DIR / "data/ground_truth/sample_form_true_data"
REPORT_PATH = BASE_DIR / "outputs/reports/ocr_accuracy_report.txt"

## Helper Functions

In [18]:
def load_json(path: Path) -> Tuple[Optional[Any], Optional[str]]:
    if not path.exists():
        return None, "missing"
    if path.stat().st_size == 0:
        return None, "empty"
    try:
        return json.loads(path.read_text()), None
    except json.JSONDecodeError as exc:
        return None, f"invalid_json: {exc}"


def extract_value(raw: Any) -> Any:
    if isinstance(raw, dict):
        if "checked" in raw:
            return raw.get("checked")
        if "text" in raw:
            return raw.get("text")
    return raw


def to_string(value: Any) -> str:
    if value is None:
        return ""
    return str(value)


def compare_documents(truth: Dict[str, Any], ocr: Dict[str, Any]) -> Tuple[int, int, List[str]]:
    matches = 0
    total = 0
    mismatches: List[str] = []

    for key, truth_value in truth.items():
        total += 1
        observed_raw = extract_value(ocr.get(key)) if ocr is not None else "<missing>"
        truth_str = to_string(truth_value)
        observed_str = to_string(observed_raw)

        if observed_str == truth_str:
            matches += 1
        else:
            mismatches.append(f"{key}: expected={truth_str!r}, observed={observed_str!r}")

    return matches, total, mismatches

## Build Report

In [19]:
def build_report() -> str:
    overall_matches = 0
    overall_total = 0
    evaluated_docs = 0
    skipped_missing_or_empty = 0
    skipped_other = 0
    per_doc_sections: List[str] = []

    ocr_files = sorted(p for p in OCR_DIR.glob("*.json") if p.is_file())

    for ocr_path in ocr_files:
        truth_path = TRUTH_DIR / ocr_path.name
        truth_data, truth_err = load_json(truth_path)

        if truth_err in {"missing", "empty"}:
            skipped_missing_or_empty += 1
            continue
        if truth_err is not None:
            skipped_other += 1
            per_doc_sections.append(
                f"{ocr_path.name}: skipped due to truth error ({truth_err})"
            )
            continue

        ocr_data, ocr_err = load_json(ocr_path)
        if ocr_err is not None:
            skipped_other += 1
            per_doc_sections.append(
                f"{ocr_path.name}: skipped due to OCR error ({ocr_err})"
            )
            continue

        if not isinstance(truth_data, dict):
            skipped_other += 1
            per_doc_sections.append(
                f"{ocr_path.name}: skipped because truth is not an object"
            )
            continue
        if not isinstance(ocr_data, dict):
            skipped_other += 1
            per_doc_sections.append(
                f"{ocr_path.name}: skipped because OCR is not an object"
            )
            continue

        matches, total, mismatches = compare_documents(truth_data, ocr_data)
        evaluated_docs += 1
        overall_matches += matches
        overall_total += total

        accuracy = matches / total if total else 0.0
        section_lines = [
            f"{ocr_path.name}",
            f"  accuracy: {accuracy:.4f} ({matches}/{total})",
        ]
        if mismatches:
            section_lines.append("  mismatches:")
            section_lines.extend(f"    - {line}" for line in mismatches)
        per_doc_sections.append("\n".join(section_lines))

    overall_accuracy = overall_matches / overall_total if overall_total else 0.0

    header = [
        "OCR Accuracy Report",
        "===================",
        f"Overall accuracy: {overall_accuracy:.4f} ({overall_matches}/{overall_total})",
        f"Documents evaluated: {evaluated_docs}",
        f"Documents skipped (missing/empty truth): {skipped_missing_or_empty}",
        f"Documents skipped (other issues): {skipped_other}",
        "",
        "Per-document results",
        "--------------------",
    ]

    return "\n".join(header + per_doc_sections)


def write_report(report_text: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(report_text)
    print(f"Report written to {destination}")

## Generate Report

In [20]:
report = build_report()
write_report(report, REPORT_PATH)
print(report)

Report written to /home/lex/GitHub/ai-adoption-research-and-development/Template-alignment/outputs/reports/ocr_accuracy_report.txt
OCR Accuracy Report
Overall accuracy: 0.7160 (537/750)
Documents evaluated: 10
Documents skipped (missing/empty truth): 0
Documents skipped (other issues): 0

Per-document results
--------------------
2025-11-13_135834.json
  accuracy: 0.7467 (56/75)
  mismatches:
    - checkbox_shelter_no: expected='False', observed='True'
    - checkbox_employment_changes_yes: expected='False', observed='True'
    - checkbox_school_spouse_no: expected='False', observed='True'
    - checkbox_work_souse_yes: expected='False', observed='True'
    - checkbox_moved_spouse_yes: expected='False', observed='True'
    - checkbox_moved_spouse_no: expected='False', observed='True'
    - checkbox_warrant_yes: expected='False', observed='True'
    - explain_changes: expected='Started part time job, documents attached', observed='Started part dime job, docwments a Hacked'
    - signatu